# Part 3 · 从词向量到文档向量

Part 2 得到的是每个**词**的 300 维向量，但分类器需要每条**影评**一个向量。
一条影评有 200 多个词，怎么合成一个？原教程给了两种做法：

```
方法 A  向量平均      逐维取算术平均 → 300 维稠密向量
方法 B  语义簇计数    先把词表 K-Means 聚类，再统计影评落在各簇的词数
```

方法 B（原教程称 *bag of centroids*）很有意思：它把 Word2Vec 学到的语义
相似性「折叠」回了 BoW 那种计数形式——`awful` 和 `terrible` 会落进同一簇，
于是计数向量里它们贡献到同一维。相当于给 BoW 装上了近义词合并。


## Setup


In [ ]:
import csv
import re
from html import unescape
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

HTML_TAG = re.compile(r"<[^>]+>")
NON_LETTER = re.compile(r"[^a-zA-Z]")
SENTENCE_END = re.compile(r"(?<=[.!?])\s+")
STOP_WORDS = frozenset(ENGLISH_STOP_WORDS)
RANDOM_STATE = 42


def find_data_dir() -> Path:
    for candidate in [Path("/kaggle/input/word2vec-nlp-tutorial"), Path("data"), Path("../data")]:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("请用 Add Input 挂载 word2vec-nlp-tutorial 竞赛数据")


def read_tsv(data_dir: Path, stem: str) -> pd.DataFrame:
    # quoting=QUOTE_NONE：影评正文含大量引号，按默认规则解析会把字段粘连。
    for suffix in (".tsv", ".tsv.zip"):
        path = data_dir / f"{stem}{suffix}"
        if path.exists():
            return pd.read_csv(path, header=0, delimiter="\t", quoting=csv.QUOTE_NONE)
    raise FileNotFoundError(f"{data_dir} 下找不到 {stem}")


def review_to_words(raw_review, remove_stopwords: bool = True) -> list[str]:
    text = NON_LETTER.sub(" ", HTML_TAG.sub(" ", unescape(str(raw_review))))
    words = text.lower().split()
    if remove_stopwords:
        return [w for w in words if w not in STOP_WORDS]
    return words


def clean_review(raw_review, remove_stopwords: bool = True) -> str:
    return " ".join(review_to_words(raw_review, remove_stopwords))


DATA_DIR = find_data_dir()
print("数据目录:", DATA_DIR)


In [ ]:
from gensim.models import Word2Vec

candidates = [
    Path("/kaggle/working/word2vec_300d.model"),
    Path("/kaggle/input/word2vec-300d/word2vec_300d.model"),
    Path("models/word2vec_300d.model"),
]
model_path = next((p for p in candidates if p.exists()), None)
assert model_path is not None, "请先运行 Part 2 生成词向量模型"

word2vec = Word2Vec.load(str(model_path))
print(f"载入 {model_path}：{len(word2vec.wv):,} 词 × {word2vec.wv.vector_size} 维")


In [ ]:
train = read_tsv(DATA_DIR, "labeledTrainData")
test = read_tsv(DATA_DIR, "testData")
print(f"训练 {len(train):,}，测试 {len(test):,}")


## 方法 A · 向量平均

逐维求平均。要点：跳过不在词表里的词（`min_count=40` 滤掉了低频词）；
一个词都没命中时返回零向量——这种情况极少，但不显式处理就会除零。


In [ ]:
def average_vectors(reviews, model) -> np.ndarray:
    vectors = model.wv
    features = np.zeros((len(reviews), vectors.vector_size), dtype=np.float32)
    for row, review in enumerate(reviews):
        known = [w for w in review_to_words(review) if w in vectors]
        if known:
            features[row] = np.mean(vectors[known], axis=0)
    return features


train_avg = average_vectors(train["review"], word2vec)
test_avg = average_vectors(test["review"], word2vec)
print(f"{train_avg.shape} —— 300 维，全部非零（对比 Part 1 的 5,000 维、98.6% 为 0）")


## 方法 B · 语义簇计数

对 16,000 多个词向量做 K-Means，簇数取 `词表大小 / 5`（原教程设定，
平均每簇约 5 个词）。

> 原教程用 `KMeans`，词表上万时相当慢。这里换 `MiniBatchKMeans`，
> 结果接近而快一个量级。


In [ ]:
from sklearn.cluster import MiniBatchKMeans

vectors = word2vec.wv
n_clusters = len(vectors.index_to_key) // 5
kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=RANDOM_STATE,
                         n_init=3, batch_size=1024)
assignments = kmeans.fit_predict(vectors.vectors)
word_to_cluster = dict(zip(vectors.index_to_key, assignments.tolist()))
print(f"{len(vectors.index_to_key):,} 词 → {n_clusters:,} 簇")


In [ ]:
# 抽查几个簇，看聚类是否符合直觉
from collections import defaultdict

clusters = defaultdict(list)
for word, cluster in word_to_cluster.items():
    clusters[cluster].append(word)

shown = 0
for cluster, words in sorted(clusters.items()):
    if 3 <= len(words) <= 8:
        print(f"簇 {cluster:5d}: {words}")
        shown += 1
        if shown == 10:
            break


In [ ]:
def centroid_vectors(reviews, word_to_cluster) -> np.ndarray:
    n_clusters = max(word_to_cluster.values()) + 1
    features = np.zeros((len(reviews), n_clusters), dtype=np.float32)
    for row, review in enumerate(reviews):
        for word in review_to_words(review):
            cluster = word_to_cluster.get(word)
            if cluster is not None:
                features[row, cluster] += 1
    return features


train_clu = centroid_vectors(train["review"], word_to_cluster)
test_clu = centroid_vectors(test["review"], word_to_cluster)
print(train_clu.shape)


## 对比两种表示


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split


def score(features, labels, name):
    tr_x, va_x, tr_y, va_y = train_test_split(
        features, labels, test_size=0.2, random_state=RANDOM_STATE, stratify=labels)
    clf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE)
    clf.fit(tr_x, tr_y)
    accuracy = accuracy_score(va_y, clf.predict(va_x))
    auc = roc_auc_score(va_y, clf.predict_proba(va_x)[:, 1])
    print(f"{name:16s} 维度 {features.shape[1]:>5,}   Accuracy {accuracy:.4f}   ROC-AUC {auc:.4f}")
    return {"name": name, "dim": features.shape[1], "accuracy": accuracy, "roc_auc": auc}


results = [
    score(train_avg, train["sentiment"], "向量平均"),
    score(train_clu, train["sentiment"], "语义簇计数"),
]


## 一个值得注意的结果

**语义簇计数明显好于向量平均**，原因在于它保留了「计数」这个结构：
一条影评出现 5 个负面情感词，对应维度就是 5。而向量平均把 5 个负面词和
1 个负面词平均成方向差别不大的结果——**强度信息被归一化掉了**。

更值得注意的是，如果和 Part 1 的 BoW（accuracy 0.8394）对比，会发现
**稠密向量并没有胜出**。这不是代码写错了，原教程本身也承认这一点。原因有几层：

- **平均操作抹平了信息**。200 个词向量取平均，词序、否定、强调全部消失，
  「不好看」和「好，不看」平均下来几乎一样。
- **语料太小**。7.5 万条影评训出的词向量，远不如在数十亿词上预训练的向量。
- **任务太窄**。单一领域的二分类，TF-IDF 这类稀疏特征本来就是极强的基线。

Word2Vec 真正的价值不在这个分数上，而在于**这份词向量可以拿去做别的任务**。
后来的 ELMo / BERT / GPT 沿着「预训练表示」这条路走下去，
解决的正是「平均会丢信息」这个问题——用注意力机制按上下文动态加权，
而不是简单求平均。


In [ ]:
import matplotlib.pyplot as plt

frame = pd.DataFrame(results)
fig, ax = plt.subplots(figsize=(6, 3.6))
positions = np.arange(len(frame))
ax.bar(positions - 0.2, frame["accuracy"], 0.4, label="Accuracy", color="#4c72b0")
ax.bar(positions + 0.2, frame["roc_auc"], 0.4, label="ROC-AUC", color="#dd8452")
ax.set(xticks=positions, ylim=(0.7, 1.0), title="文档向量方案对比")
ax.set_xticklabels(frame["name"])
ax.legend()
plt.tight_layout()
plt.show()


## 生成提交文件


In [ ]:
best = max(results, key=lambda r: r["roc_auc"])
print("选用:", best["name"])
train_features, test_features = (
    (train_avg, test_avg) if best["name"] == "向量平均" else (train_clu, test_clu))

final = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE)
final.fit(train_features, train["sentiment"])
submission = pd.DataFrame({"id": test["id"], "sentiment": final.predict(test_features)})

output = Path("/kaggle/working/submission.csv") if Path("/kaggle/working").exists() else Path("submission.csv")
submission.to_csv(output, index=False, quoting=csv.QUOTE_NONE)
print(f"已写出 {output}")
submission.head()


## 三部分串起来看

| | 表示 | 维度 | 每一维的含义 |
|---|---|---|---|
| Part 1 | Bag of Words | 5,000 稀疏 | 某个具体词的次数 |
| Part 2 | Word2Vec 词向量 | 300 稠密 | 不可单独解读 |
| Part 3 | 影评向量（平均 / 簇计数） | 300 / ~3,300 | 同上 / 某个语义簇的次数 |

从 Part 1 到 Part 3，走完的是 NLP 表示学习最关键的一次转向：
**从「统计某个词出现几次」到「把语义编码进连续空间的坐标」**。
今天大模型的 Embedding 层做的还是同一件事，只是向量由 Transformer
按上下文动态生成，而不再是每个词一张固定的查找表。
